# Parallel Cricket Workflow using LangGraph
This notebook defines a LangGraph workflow that takes cricket match statistics and calculates the strike rate, dot ball percentage, boundary percentage, and a summary in parallel.

In [ ]:
%%capture
!pip install langgraph langchain-openai

In [ ]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, START, END

class CricketState(TypedDict):
    player_name: str
    runs: int
    balls: int
    dot_balls: int
    boundaries: int
    
    # Computed metrics
    strike_rate: Optional[float]
    dot_ball_percentage: Optional[float]
    boundary_percentage: Optional[float]
    summary: Optional[str]


In [ ]:
# Define the nodes

def compute_strike_rate(state: CricketState):
    runs = state["runs"]
    balls = state["balls"]
    if balls > 0:
        sr = (runs / balls) * 100
    else:
        sr = 0.0
    return {"strike_rate": round(sr, 2)}

def compute_dot_ball_percentage(state: CricketState):
    dot_balls = state["dot_balls"]
    balls = state["balls"]
    if balls > 0:
        dbp = (dot_balls / balls) * 100
    else:
        dbp = 0.0
    return {"dot_ball_percentage": round(dbp, 2)}

def compute_boundary_percentage(state: CricketState):
    boundaries = state["boundaries"]
    balls = state["balls"]
    if balls > 0:
        bp = (boundaries / balls) * 100
    else:
        bp = 0.0
    return {"boundary_percentage": round(bp, 2)}

def generate_summary(state: CricketState):
    return {
        "summary": f"Player {state['player_name']} scored {state['runs']} runs off {state['balls']} balls, including {state['boundaries']} boundaries and {state['dot_balls']} dot balls."
    }


In [ ]:
# Build the graph
workflow = StateGraph(CricketState)

# Add nodes
workflow.add_node("strike_rate", compute_strike_rate)
workflow.add_node("dot_ball_percent", compute_dot_ball_percentage)
workflow.add_node("boundary_percent", compute_boundary_percentage)
workflow.add_node("summary", generate_summary)

# Connect nodes in parallel
workflow.add_edge(START, "strike_rate")
workflow.add_edge(START, "dot_ball_percent")
workflow.add_edge(START, "boundary_percent")
workflow.add_edge(START, "summary")

workflow.add_edge("strike_rate", END)
workflow.add_edge("dot_ball_percent", END)
workflow.add_edge("boundary_percent", END)
workflow.add_edge("summary", END)

# Compile the workflow
app = workflow.compile()


In [ ]:
from IPython.display import Image, display
try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Could not draw the graph:", e)


In [ ]:
# Execute the workflow
input_state = {
    "player_name": "Virat Kohli",
    "runs": 82,
    "balls": 53,
    "dot_balls": 12,
    "boundaries": 10
}

final_state = app.invoke(input_state)

print("Workflow Execution Results:\n")
print(f"Strike Rate: {final_state['strike_rate']}%")
print(f"Dot Ball Percentage: {final_state['dot_ball_percentage']}%")
print(f"Boundary Percentage: {final_state['boundary_percentage']}%")
print(f"Summary: {final_state['summary']}")
